In [ ]:
# 1. Setup (same as notebook 03)
import sys; sys.path.append("../src")
import os; os.makedirs("../models", exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
from data_utils import scan_dataset
from preprocessing import *
from classifier import *

df, _ = scan_dataset("../data")
val_ok = set(df[(df.folder == "Val") & ~df.corrupted & (df.n_bad_boxes == 0)].path)
split_df = build_table("../data", val_ok_paths=val_ok)
class_names = get_class_names()

train_ds = make_dataset(split_df[split_df.split == "train"], class_names, train=True)
val_ds   = make_dataset(split_df[split_df.split == "val"],   class_names, train=False)

In [ ]:
# 2. Two configurations (use the model you chose in Étape 3)
BACKBONE = "mobilenetv2"

# A: baseline, only data augmentation
model_a, hist_a = run_config("config_A", BACKBONE, train_ds, val_ds,
                             dropout=0.0, l2=0.0, use_callbacks=False)

# B: all techniques
model_b, hist_b = run_config("config_B", BACKBONE, train_ds, val_ds,
                             dropout=0.3, l2=1e-4, use_callbacks=True)

In [ ]:
# 3. Learning curves
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for row, (name, h) in enumerate([("A: baseline", hist_a), ("B: all techniques", hist_b)]):
    for col, (k, title) in enumerate([("loss", "Loss"), ("accuracy", "Accuracy")]):
        ax = axes[row][col]
        ax.plot(h[k], label="train"); ax.plot(h["val_" + k], label="val")
        ax.set_title(f"{name} - {title}"); ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# 4. Compare on validation
pd.DataFrame([
    {"config": "A", "val_loss": model_a.evaluate(val_ds, verbose=0)[0], "val_acc": model_a.evaluate(val_ds, verbose=0)[1]},
    {"config": "B", "val_loss": model_b.evaluate(val_ds, verbose=0)[0], "val_acc": model_b.evaluate(val_ds, verbose=0)[1]},
]).round(4)